# 04 — Gold Layer Explorer
> **Uso pessoal / QA** — não faz parte do pipeline de produção. Não commitar.
> Roda no Microsoft Fabric com `lh_yellow_taxi` attachado como lakehouse padrão.

### Tabelas cobertas
| Camada | Tabela | Tipo |
|--------|--------|------|
| Gold | `fact_trips` | Fato (115.7M) |
| Gold | `dim_date` | Dimensão (1096) |
| Gold | `dim_time` | Dimensão (24) |
| Gold | `dim_zone` | Dimensão (265) |
| Gold | `dim_zone_dropoff` | Dimensão (265) |
| Gold | `dim_vendor` | Dimensão (5) |
| Gold | `dim_ratecode` | Dimensão (7) |
| Gold | `dim_payment` | Dimensão (7) |
| Gold | `discard_reasons` | Lookup (4) |
| Silver | `silver_yellow_trips_clean` | Fato clean (115.7M) |

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────────
from pyspark.sql import functions as F

SAMPLE_N = 10

ALL_TABLES = [
    "fact_trips",
    "dim_date",
    "dim_time",
    "dim_zone",
    "dim_zone_dropoff",
    "dim_vendor",
    "dim_ratecode",
    "dim_payment",
    "discard_reasons",
    "silver_yellow_trips_clean",
]

---
## 1 · Resumo Geral — Row counts

In [ ]:
# ── Tabela rápida: linhas + colunas de cada tabela ─────────────────────────
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType

summary_rows = []
print(f"{'Tabela':<30} {'Linhas':>15} {'Colunas':>10}  Chave primária")
print("-" * 80)

pk_map = {
    "fact_trips":               "date_key + pu_zone_key + time_key (comp.)",
    "dim_date":                 "date_key",
    "dim_time":                 "time_key",
    "dim_zone":                 "zone_key",
    "dim_zone_dropoff":         "dropoff_zone_key",
    "dim_vendor":               "vendor_key",
    "dim_ratecode":             "ratecode_key",
    "dim_payment":              "payment_key",
    "discard_reasons":          "reason",
    "silver_yellow_trips_clean":"pickup_datetime (natural)",
}

for tbl in ALL_TABLES:
    try:
        df   = spark.table(tbl)
        rows = df.count()
        cols = len(df.columns)
        pk   = pk_map.get(tbl, "—")
        print(f"{tbl:<30} {rows:>15,} {cols:>10}  {pk}")
        summary_rows.append((tbl, rows, cols))
    except Exception as e:
        print(f"{tbl:<30} {'ERRO':>15}  →  {e}")

---
## 2 · Dimensões pequenas — display completo
> `dim_time` (24), `dim_vendor` (5), `dim_ratecode` (7), `dim_payment` (7), `discard_reasons` (4)

In [ ]:
# ── dim_time ───────────────────────────────────────────────────────────────
print("=" * 60)
print("  DIM_TIME (24 linhas — 1 por hora)")
print("=" * 60)
display(spark.table("dim_time").orderBy("time_key"))

In [ ]:
# ── dim_vendor ─────────────────────────────────────────────────────────────
print("=" * 60)
print("  DIM_VENDOR (5 linhas)")
print("=" * 60)
display(spark.table("dim_vendor").orderBy("vendor_key"))

In [ ]:
# ── dim_ratecode ───────────────────────────────────────────────────────────
print("=" * 60)
print("  DIM_RATECODE (7 linhas)")
print("=" * 60)
display(spark.table("dim_ratecode").orderBy("ratecode_key"))

In [ ]:
# ── dim_payment ────────────────────────────────────────────────────────────
print("=" * 60)
print("  DIM_PAYMENT (7 linhas)")
print("=" * 60)
display(spark.table("dim_payment").orderBy("payment_key"))

In [ ]:
# ── discard_reasons ────────────────────────────────────────────────────────
print("=" * 60)
print("  DISCARD_REASONS (4 linhas — registros descartados na silver)")
print("=" * 60)
display(
    spark.table("discard_reasons")
    .withColumn("pct_of_bronze", F.round(F.col("rows") / 119207946 * 100, 3))
    .orderBy(F.desc("rows"))
)

---
## 3 · dim_date — Calendário + Weather

In [ ]:
df_date = spark.table("dim_date")

# Schema
print("[Schema — dim_date]")
df_date.printSchema()

# Range e totais
print("\n[Range de datas]")
display(df_date.agg(
    F.min("full_date").alias("data_min"),
    F.max("full_date").alias("data_max"),
    F.count("*").alias("total_dias"),
    F.sum("is_weekend").alias("fins_de_semana"),
    F.sum("is_extreme_weather").alias("dias_clima_extremo")
))

# Sample
print(f"\n[Amostra — {SAMPLE_N} linhas]")
display(df_date.orderBy("full_date").limit(SAMPLE_N))

In [ ]:
# Distribuição por weather_category e season
print("[dim_date — weather_category]")
display(
    spark.table("dim_date")
    .groupBy("weather_category")
    .agg(
        F.count("*").alias("dias"),
        F.round(F.count("*") * 100.0 / 1096, 1).alias("pct")
    )
    .orderBy(F.desc("dias"))
)

print("\n[dim_date — season × year]")
display(
    spark.table("dim_date")
    .groupBy("season", "year")
    .agg(F.count("*").alias("dias"))
    .orderBy("year", "season")
)

---
## 4 · dim_zone — Zonas + ACS Demographics

In [ ]:
df_zone = spark.table("dim_zone")

print("[Schema — dim_zone]")
df_zone.printSchema()

print("\n[Breakdown por borough]")
display(
    df_zone.groupBy("borough", "service_zone")
    .agg(
        F.count("*").alias("zonas"),
        F.first("total_population").alias("populacao"),
        F.first("median_household_income").alias("renda_median_usd"),
        F.first("population_density_per_sq_mi").alias("densidade_pop_sqmi")
    )
    .orderBy(F.desc("populacao"))
)

print(f"\n[Amostra — {SAMPLE_N} linhas]")
display(df_zone.orderBy("zone_key").limit(SAMPLE_N))

In [ ]:
# dim_zone_dropoff — confirmar que é espelho de dim_zone
df_zdo  = spark.table("dim_zone_dropoff")
df_zone = spark.table("dim_zone")

print("[dim_zone_dropoff — amostra]")
display(df_zdo.orderBy("dropoff_zone_key").limit(SAMPLE_N))

count_match = df_zone.count() == df_zdo.count()
print(f"\ndim_zone ({df_zone.count()}) == dim_zone_dropoff ({df_zdo.count()})? → {count_match}")

---
## 5 · fact_trips — Amostra, Schema e Distribuições-chave

In [ ]:
df_fact = spark.table("fact_trips")

print("[Schema — fact_trips]")
df_fact.printSchema()

print(f"\n[Amostra — {SAMPLE_N} linhas]")
display(df_fact.limit(SAMPLE_N))

In [ ]:
# Distribuição por duration_class (bucket de lead time)
print("[fact_trips — duration_class (lead time bucket)]")
display(
    df_fact.groupBy("duration_class")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.count("*") * 100.0 / 115756175, 2).alias("pct")
    )
    .orderBy("duration_class")
)

# Distribuição por day_part
print("\n[fact_trips — day_part]")
display(
    df_fact.groupBy("day_part")
    .agg(F.count("*").alias("trips"),
         F.round(F.sum("total_amount"), 0).alias("revenue_usd"))
    .orderBy(F.desc("trips"))
)

# Volume por year/month
print("\n[fact_trips — volume por year × month]")
display(
    df_fact.groupBy("year", "month")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.sum("total_amount"), 0).alias("revenue_usd")
    )
    .orderBy("year", "month")
)

In [ ]:
# Estatísticas das medidas numéricas
print("[fact_trips — describe medidas numéricas]")
measure_cols = [
    "passenger_count", "trip_distance_mi", "duration_min", "avg_speed_mph",
    "fare_amount", "tip_amount", "total_amount", "congestion_surcharge"
]
display(df_fact.select(measure_cols).describe())

---
## 6 · QA — Nulls e Orphan Keys

In [ ]:
# Nulls por coluna (fact_trips)
print("[fact_trips — null counts por coluna]")
null_df = df_fact.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_fact.columns
])
display(null_df)

In [ ]:
# Orphan FK check
df_zone = spark.table("dim_zone")
df_vend = spark.table("dim_vendor")
df_pay  = spark.table("dim_payment")
df_date = spark.table("dim_date")

orphan_pu   = df_fact.join(df_zone, df_fact.pu_zone_key  == df_zone.zone_key,  "left_anti").count()
orphan_do   = df_fact.join(df_zone, df_fact.do_zone_key  == df_zone.zone_key,  "left_anti").count()
orphan_vend = df_fact.join(df_vend, df_fact.vendor_key   == df_vend.vendor_key, "left_anti").count()
orphan_pay  = df_fact.join(df_pay,  df_fact.payment_key  == df_pay.payment_key, "left_anti").count()
orphan_date = df_fact.join(df_date, df_fact.date_key     == df_date.date_key,   "left_anti").count()

print(f"Orphans pu_zone_key   : {orphan_pu:,}")
print(f"Orphans do_zone_key   : {orphan_do:,}")
print(f"Orphans vendor_key    : {orphan_vend:,}")
print(f"Orphans payment_key   : {orphan_pay:,}")
print(f"Orphans date_key      : {orphan_date:,}")
print()
all_clean = all(v == 0 for v in [orphan_pu, orphan_do, orphan_vend, orphan_pay, orphan_date])
print(f"✅ Integridade referencial OK? → {all_clean}")

---
## 7 · Análises Extras — Bom para estudar antes da apresentação

In [ ]:
# ── A. Receita por borough × year (business sanity) ────────────────────────
print("[Receita por borough × year — Manhattan deve dominar]")
display(
    df_fact
    .join(spark.table("dim_zone"), df_fact.pu_zone_key == spark.table("dim_zone").zone_key, "left")
    .join(spark.table("dim_date"), df_fact.date_key    == spark.table("dim_date").date_key,  "left")
    .groupBy("borough", "year")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.sum("total_amount"), 0).alias("revenue_usd"),
        F.round(F.avg("total_amount"), 2).alias("avg_fare"),
        F.round(F.avg("duration_min"), 1).alias("avg_lead_time_min")
    )
    .orderBy("borough", "year")
)

In [ ]:
# ── B. Heatmap hora × dia da semana (demanda) ──────────────────────────────
# Usado na tela Demand Deep Dive do Power BI
df_time = spark.table("dim_time")
df_date = spark.table("dim_date")

print("[Trips por hora × dia da semana — base do heatmap]")
display(
    df_fact
    .join(df_date, "date_key")
    .join(df_time, df_fact.time_key == df_time.time_key, "left")
    .groupBy("day_name", "day_order_mon", "hour", "hour_label", "is_rush_hour")
    .agg(F.count("*").alias("trips"))
    .orderBy("day_order_mon", "hour")
)

In [ ]:
# ── C. Impacto do clima nas viagens ───────────────────────────────────────
# Valida a narrativa de Supply Chain: eventos externos (clima) afetam demanda
print("[Trips × clima — impacto de dias extremos vs normais]")
display(
    df_fact
    .join(spark.table("dim_date"), "date_key")
    .groupBy("weather_category", "is_extreme_weather")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("total_amount"), 2).alias("avg_fare"),
        F.round(F.avg("duration_min"), 1).alias("avg_lead_time"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip")
    )
    .orderBy(F.desc("total_trips"))
)

In [ ]:
# ── D. Top 15 pares pickup-dropoff (volume) ────────────────────────────────
df_zpu = spark.table("dim_zone").select(
    F.col("zone_key").alias("pu_zone_key"),
    F.col("zone_name").alias("pu_zone"),
    F.col("borough").alias("pu_borough")
)
df_zdo = spark.table("dim_zone_dropoff").select(
    F.col("dropoff_zone_key").alias("do_zone_key"),
    F.col("dropoff_zone_name").alias("do_zone"),
    F.col("dropoff_borough").alias("do_borough")
)

print("[Top 15 pares pickup → dropoff]")
display(
    df_fact
    .join(df_zpu, "pu_zone_key", "left")
    .join(df_zdo, "do_zone_key", "left")
    .groupBy("pu_zone", "pu_borough", "do_zone", "do_borough")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("total_amount"), 2).alias("avg_fare")
    )
    .orderBy(F.desc("trips"))
    .limit(15)
)

In [ ]:
# ── E. Taxa de gorjeta por tipo de pagamento ──────────────────────────────
# Insight: cartão tem gorjeta, dinheiro não → viés na média de tip
print("[Tip rate por payment_type — gorjeta existe só em cartão]")
display(
    df_fact
    .join(spark.table("dim_payment"), "payment_key")
    .groupBy("payment_type_name")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
        F.round(
            F.sum(F.when(F.col("tip_amount") > 0, 1).otherwise(0)) * 100.0 / F.count("*"), 1
        ).alias("pct_trips_com_gorjeta"),
        F.round(F.sum("tip_amount") / F.sum("fare_amount") * 100, 1).alias("tip_rate_pct")
    )
    .orderBy(F.desc("trips"))
)

In [ ]:
# ── F. YoY: variação de volume e receita ──────────────────────────────────
# Insight que foi para o Executive Overview (+5.7% trips YoY)
from pyspark.sql.window import Window

yearly = (
    df_fact
    .groupBy("year")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.sum("total_amount"), 0).alias("revenue_usd"),
        F.round(F.avg("total_amount"), 2).alias("avg_fare"),
        F.round(F.avg("duration_min"), 1).alias("avg_lead_time_min")
    )
)

w = Window.orderBy("year")
yearly_yoy = (
    yearly
    .withColumn("trips_prev",   F.lag("trips").over(w))
    .withColumn("rev_prev",     F.lag("revenue_usd").over(w))
    .withColumn("yoy_trips_pct",
        F.round((F.col("trips") - F.col("trips_prev")) / F.col("trips_prev") * 100, 2))
    .withColumn("yoy_rev_pct",
        F.round((F.col("revenue_usd") - F.col("rev_prev")) / F.col("rev_prev") * 100, 2))
    .drop("trips_prev", "rev_prev")
    .orderBy("year")
)

print("[YoY — trips e receita por ano]")
display(yearly_yoy)

In [ ]:
# ── G. Anomalias por borough × year ──────────────────────────────────────
# Tela Data Quality do Power BI — confirmar que o rate ~0.16% bate
print("[Anomalias (is_anomalous=1) por borough × year]")
display(
    df_fact
    .filter(F.col("is_anomalous") == 1)
    .join(spark.table("dim_zone"), df_fact.pu_zone_key == spark.table("dim_zone").zone_key, "left")
    .groupBy("borough", "year")
    .agg(
        F.count("*").alias("anomalous_trips"),
        F.round(F.avg("avg_speed_mph"), 1).alias("avg_speed_mph"),
        F.round(F.avg("trip_distance_mi"), 2).alias("avg_dist_mi")
    )
    .orderBy("borough", "year")
)

# Taxa global
total_trips    = df_fact.count()
anomaly_trips  = df_fact.filter(F.col("is_anomalous") == 1).count()
print(f"\nAnomaly rate global: {anomaly_trips / total_trips * 100:.4f}%  ({anomaly_trips:,} de {total_trips:,})")

---
> **Fim do explorer.**  
> Notebook local/pessoal — não faz parte do pipeline de produção e não precisa ser commitado.